## Baseline 3 — Windforce OOP Pipeline (Full Version)

Windforce 패키지의 클래스를 활용해 **데이터 로드 → 격자 전처리 → 그룹 공간집계 → 모형 학습 → 제출** 까지  
전 과정을 재현 가능하게 구성한 노트북이다.

| Step | 내용 |
|------|------|
| 1 | 파일 경로 확인 (`WindforceDataLoader.check_paths`) |
| 2 | 데이터 로드 및 구조 확인 |
| 3 | 터빈 메타 준비 — KPX 그룹·좌표 파싱 (`kpx_info`) |
| 4 | 데이터 전처리 및 시각화 — LDAPS·GFS 격자 변환 + IDW 그룹 집계 |
| 5 | SCADA 파워 커브 시각화 |
| 6 | 평가지표 계산 (`EvaluationMetrics`) + 손실함수 확인 (`ScoreLossFunction`) |
| 7 | 그룹별 모형 학습 (`WindforceDatasetBuilder` + `GroupExperimentRunner`) |
| 8 | 스키마 검증 + 제출 CSV 저장 |

In [12]:
import os
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import statsmodels.api as stats
# 기본 데이터 분석·시각화 라이브러리

In [13]:
import torch
import torch.nn as nn
# pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
# GPU 없는 환경에서는 CPU 전용 빌드를 설치한다

In [14]:
ROOT = "/Users/ksydata/WINDFORCE"
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
# Windforce 패키지 탐색 경로를 sys.path에 추가

from Windforce import (
    WindforceDataLoader,
    FeatureEngineer,
    LDAPSFeatureEngineer,
    GFSFeatureEngineer,
    EvaluationMetrics,
    RATED_CAPACITY_KW,
    TIME_STEP_HOURS,
    ScoreLossFunction,
)
# Windforce 루트 패키지에서 주요 클래스·상수 import

from Windforce.Preprocessing import (
    PreprocessorFactory,
    SCADAFeatureEngineer,
)
# Preprocessing 서브패키지에서 전처리기 import

from Windforce.utils import (
    KPX_GROUPS,
    PreprocessingError,
    compute_haversine_distance,
    compute_group_weight,
    transform_to_group_feature,
    MAX_INTERP_STEPS,
    interpolate_short_gap,
    select_group,
    transform_cyclic_time,
    compute_wind_speed,
    compute_wind_direction,
    compute_wind_shear,
    calculate_air_density,
    calculate_wind_power_density,
)
# utils 서브패키지에서 공통 유틸리티 import

from Windforce.Modeling import (
    LSTMPipeline,
    GroupExperimentRunner,
    BaselineModels,
)
# Modeling 서브패키지에서 학습·실험 관련 클래스 import


TypeError: unsupported operand type(s) for |: 'type' and 'type'

In [ ]:
pd.set_option("display.max_columns", None)
# 열 생략 없이 모든 컬럼 표시
pd.options.display.float_format = "{:,.6f}".format
# 지수 표기(e) 대신 실수 6자리 고정 소수점으로 출력

GROUPS  = [1, 2, 3]
# KPX 평가 그룹 번호 목록
SEQ_LEN = 24
# LSTM 입력 시퀀스 길이 (최근 24시간)
SEED    = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
# 재현성 확보를 위해 난수 시드 고정

## Step 1. 파일 경로 확인
모든 경로 내 파일이 실제로 존재하는지 확인한다.

In [ ]:
loader = WindforceDataLoader()
# WindforceDataLoader 인스턴스 생성 (root 기본값 = "/Users/ksydata/WINDFORCE")
loader.check_paths()
# 등록된 모든 파일의 존재 여부를 ✅/❌ 로 출력

## Step 2. 데이터 로드 및 구조 확인

- 풍력발전량 예측에서 입력으로 봐야 할 기상 변수는 무엇인가?
- 시간 단위 예측값 제출에서 실수하기 쉬운 포맷은 무엇인가?
- NMAE와 정산금획득률(FICR) 중 어느 지표가 실험 로그에 먼저 들어가야 하는가?

In [ ]:
print("=== SCADA VESTAS ===")
print(loader["scada_vestas_train"].info())
# VESTAS 제작사 12개 터빈의 10분 단위 실측 데이터 (풍속·풍향·발전량)
print("\n=== SCADA UNISON ===")
print(loader["scada_unison_train"].info())
# UNISON 제작사 5개 터빈의 10분 단위 실측 데이터

In [ ]:
print("=== train_labels (첫 5행) ===")
print(loader["train_labels"].head(5))
# kst_dtm: 발전량 집계 구간의 종료 시각 (1시간 단위)
# kpx_group_1/2/3: KPX 그룹별 시간당 실제 발전량 (kWh)
print()
print("=== sample_submission (첫 5행) ===")
print(loader["sample_submission"].head(5))
# 제출 양식: 2025년 전체 기간(8,760시간)의 3개 그룹 발전량 예측값을 채워 제출

In [ ]:
loader["kpx_info"]
# KPX 그룹·터빈 메타 정보 (단계, 제작사, 모델명, 호기, 좌표, KPX그룹, 허브높이, 로터지름, 설비용량)

In [ ]:
loader["ldaps_train"].head(2)
# LDAPS 학습 예보 데이터 첫 2행:
# forecast_kst_dtm 하나당 grid_id 16개 행 존재 (약 1.5km 해상도)

In [ ]:
loader["gfs_train"].head(2)
# GFS 학습 예보 데이터 첫 2행:
# forecast_kst_dtm 하나당 grid_id 9개 행 존재 (약 0.25도 해상도, 80·100m 고도 바람 포함)

## Step 3. 터빈 메타 준비

`LDAPSFeatureEngineer.transformToGroupIDW()` 와 `GFSFeatureEngineer.transformToGroupIDW()` 는  
`turbine_meta` DataFrame을 받아 16격자(또는 9격자)를 KPX 그룹별 역거리가중(IDW)으로 집계한다.

필요 컬럼: `group` (KPX그룹 번호), `lat`, `lon` (터빈 좌표), `cap_kw` (개별 터빈 설비용량 kW)

In [ ]:
import re

def parseDMS(coord_str: str) -> tuple[float, float]:
    """도분초(DMS) 형식 문자열을 십진수(DD) 위도·경도로 변환하는 함수

    Args:
        - coord_str: '37°16\'55.61"N 128°57\'02.10"E' 형태의 문자열

    Returns:
        - (위도, 경도) 십진수 튜플
    """
    # 숫자 토큰과 방향(N/S/E/W) 분리
    tokens = re.findall(r"[\d.]+|[NSEW]", coord_str.replace('""', '"'))
    # 위도(lat): 첫 번째 도/분/초/방향 4개 토큰
    lat_d, lat_m, lat_s, lat_dir = float(tokens[0]), float(tokens[1]), float(tokens[2]), tokens[3]
    # 경도(lon): 두 번째 도/분/초/방향 4개 토큰
    lon_d, lon_m, lon_s, lon_dir = float(tokens[4]), float(tokens[5]), float(tokens[6]), tokens[7]

    lat = lat_d + lat_m / 60 + lat_s / 3600
    lon = lon_d + lon_m / 60 + lon_s / 3600
    # 남반구(S)·서경(W)은 음수로 변환 (이 데이터는 국내 북위·동경이라 발동 안 됨)
    if lat_dir == "S":
        lat = -lat
    if lon_dir == "W":
        lon = -lon
    return lat, lon


kpx_raw = loader["kpx_info"].copy()
# kpx_info 로드 (터빈 1기 = 1행)

coords = kpx_raw["좌표(Google)"].apply(parseDMS)
# DMS 좌표 문자열을 (위도, 경도) 튜플로 변환

kpx_raw["lat"] = coords.apply(lambda x: x[0])
kpx_raw["lon"] = coords.apply(lambda x: x[1])
# 십진도 위도·경도를 별도 컬럼으로 분리

turbine_meta = kpx_raw[["KPX그룹", "lat", "lon", "설비용량(MW)"]].rename(
    columns={"KPX그룹": "group", "설비용량(MW)": "cap_kw"}
).copy()
turbine_meta["cap_kw"] = turbine_meta["cap_kw"] * 1000
# MW → kW 단위 변환
turbine_meta = turbine_meta.reset_index(drop=True)

print(turbine_meta)
# 터빈별 KPX그룹·위도·경도·설비용량(kW) 확인

---
## Step 4. 데이터 전처리 및 시각화

### 4-1. LDAPS 격자 전처리 + KPX 그룹 공간집계 (IDW)

```
ldaps_raw (16격자 × 시각)
  → LDAPSFeatureEngineer.transform()          8단계 파이프라인 (격자별 파생변수)
  → LDAPSFeatureEngineer.transformToGroupIDW() 16격자 → 3그룹 역거리가중 집계
  → ldaps_group_df (그룹 × 시각 표)
```

In [ ]:
ldaps_fe = LDAPSFeatureEngineer()
# 그룹 지정 없이 생성 → transform() 후 전체 16격자 반환

print("LDAPS 격자 전처리 중 (16격자 × 26,304 시각)...")
ldaps_grid = ldaps_fe.transform(loader["ldaps_train"])
# 8단계 파이프라인: 컬럼명 통일 → 시간 구조화 → 물리 한계 플래그 →
#   보간 → 파생변수(풍속·공기밀도·풍력에너지밀도) → 급변 플래그 → 주기 인코딩 → selectGroup

print(f"ldaps_grid shape: {ldaps_grid.shape}")
ldaps_grid.head(2)

In [ ]:
print("LDAPS IDW 그룹 집계 중...")
ldaps_group_df = ldaps_fe.transformToGroupIDW(ldaps_grid, turbine_meta)
# 16격자 → 3그룹 역거리가중 집계
# turbine_meta: group / lat / lon / cap_kw 컬럼 필수

print(f"ldaps_group_df shape: {ldaps_group_df.shape}")
# 결과: (그룹 수 × 시각 수) 행 = 3 × 26,304 = 78,912행
ldaps_group_df.head(3)

In [ ]:
# ─── GFS 격자 전처리 + KPX 그룹 공간집계 ───
gfs_fe = GFSFeatureEngineer()
# 그룹 지정 없이 생성 → 전체 9격자 처리

print("GFS 격자 전처리 중 (9격자 × 26,304 시각)...")
gfs_grid = gfs_fe.transform(loader["gfs_train"])
# 컬럼명 통일(gfs_ 접두사) → 시간 구조화 → 물리 한계 플래그 →
#   다층 바람(10·80·100m·PBL·850·700·500hPa) 파생 → 전단지수·허브높이 외삽 → 주기 인코딩

print(f"gfs_grid shape: {gfs_grid.shape}")
gfs_grid.head(2)

In [ ]:
print("GFS IDW 그룹 집계 중...")
gfs_group_df = gfs_fe.transformToGroupIDW(gfs_grid, turbine_meta)
# 9격자 → 3그룹 역거리가중 집계

print(f"gfs_group_df shape: {gfs_group_df.shape}")
gfs_group_df.head(3)

### 4-2. SCADA 파워 커브 시각화

SCADA 실측 데이터에서 풍속(ws)·발전량(power_kw10m)의 관계를 확인해  
모델이 학습해야 하는 비선형 패턴(S자 파워 커브)을 시각적으로 검증한다.

In [ ]:
# VESTAS 12개 호기 기초 통계 확인
for turbine in ["vestas_wtg01", "vestas_wtg02", "vestas_wtg03",
                "vestas_wtg04", "vestas_wtg05", "vestas_wtg06",
                "vestas_wtg07", "vestas_wtg08", "vestas_wtg09",
                "vestas_wtg10", "vestas_wtg11", "vestas_wtg12"]:
    print(turbine, "\n",
          loader["scada_vestas_train"].filter(like=turbine).describe())
    # 마이너스 발전량 또는 설비용량(3,600 kW) 초과값에 유의
    # → 이상치 처리 필요 (정비·고장 구간)

In [ ]:
# UNISON 5개 호기 기초 통계 확인
for turbine in ["unison_wtg01", "unison_wtg02", "unison_wtg03",
                "unison_wtg04", "unison_wtg05"]:
    print(turbine, "\n",
          loader["scada_unison_train"].filter(like=turbine).describe())

In [ ]:
# UNISON 터빈별 3D 산점도 (풍속 × 풍향 × 발전량)
target_df = loader["scada_unison_train"]
# 시각화 대상 SCADA 데이터셋 선택

# 터빈명(unison_wtg01~05) 자동 추출
turbine_names = []
for col in target_df.columns:
    if col.startswith("unison_wtg"):
        base = col.replace("_ws", "").replace("_wd", "").replace("_power_kw10m", "")
        if base not in turbine_names:
            turbine_names.append(base)
print(f"터빈 수: {len(turbine_names)}")

# 첫 번째 터빈만 3D 산점도 시각화
for t in turbine_names:
    ws, wd, pw = f"{t}_ws", f"{t}_wd", f"{t}_power_kw10m"
    if {ws, wd, pw}.issubset(target_df.columns):
        fig = px.scatter_3d(
            target_df, x=ws, y=wd, z=pw,
            color=pw,
            color_continuous_scale="Viridis",
            title=f"[{t}] Wind Speed × Direction × Power",
            labels={ws: "Wind Speed (m/s)", wd: "Wind Direction (°)", pw: "Power (kW)"},
            opacity=0.7,
        )
        fig.update_layout(margin=dict(l=0, r=0, b=0, t=40))
        fig.show()
        break
        # 렌더링 시간 절약을 위해 첫 번째 터빈만 표시

In [ ]:
# LOWESS 트렌드라인으로 파워 커브 확인
for t in turbine_names:
    ws, pw = f"{t}_ws", f"{t}_power_kw10m"
    if {ws, pw}.issubset(target_df.columns):
        curve_df = target_df[[ws, pw]].dropna()
        curve_df = curve_df[(curve_df[ws] > 0.5) & (curve_df[pw] >= 0)]
        # 풍속 0.5 m/s 이하(정지 구간) 및 음수 발전량(이상치) 제거
        if len(curve_df) > 3000:
            curve_df = curve_df.sample(3000, random_state=SEED)
            # 시각화 속도를 위해 3,000개 샘플링

        fig = px.scatter(
            curve_df, x=ws, y=pw,
            trendline="lowess",
            trendline_color_override="red",
            title=f"[{t}] Power Curve (Wind Speed vs Power, LOWESS)",
            labels={ws: "Wind Speed (m/s)", pw: "Power (kW)"},
            opacity=0.4,
            template="plotly_white",
        )
        fig.update_layout(margin=dict(l=0, r=0, b=0, t=40))
        fig.show()
        break
        # S자 곡선(컷인→정격→컷아웃) 패턴이 보이면 LSTM이 학습 가능한 비선형 구조 확인

---
## Step 5. 평가지표 계산 및 손실함수 확인

### 대회 규정 정리

| 지표 | 정의 |
|------|------|
| **1-NMAE** | 그룹별 `NMAE_g = mean(|pred-actual| / 설비용량)`, 3그룹 평균 후 `1 - NMAE` 로 변환. **클수록 좋음** |
| **FICR** | 시간별 오차율 구간(≤6%: 4원, 6~8%: 3원, >8%: 0원)으로 정산금 계산 후 최대 정산금 대비 비율. **클수록 좋음** |
| **총점** | `0.5 × (1-NMAE) + 0.5 × FICR` |

### 손실함수 설계

- `loss = 1 - 총점 = 0.5×NMAE + 0.5×(1-FICR)`
- FICR의 계단함수(4/3/0원)를 시그모이드 2개로 완화해 미분 가능하게 근사 (`ScoreLossFunction`)

In [ ]:
# EvaluationMetrics 인스턴스 생성 및 더미 예측값으로 동작 검증
metrics = EvaluationMetrics(
    rated_capacity_kw=RATED_CAPACITY_KW,
    time_step_hours=TIME_STEP_HOURS,
)
# RATED_CAPACITY_KW = {1: 21600, 2: 21600, 3: 21000} (kW)
# TIME_STEP_HOURS   = 1.0 (시간 단위 발전량)

labels_df = loader["train_labels"]
print("train_labels 컬럼:", labels_df.columns.tolist())
print(labels_df.head(3))

# 더미 예측: 모든 시간대 설비용량의 5% 발전 가정
rng = np.random.default_rng(SEED)
for g in GROUPS:
    actual = labels_df[f"kpx_group_{g}"].dropna().values
    pred   = actual * rng.uniform(0.8, 1.2, size=len(actual))  # ±20% 노이즈 추가
    nmae   = metrics.evaluateNMAE(pred, actual, g)
    ficr   = metrics.evaluateFICR(pred, actual, g)
    score  = 0.5 * (1 - nmae) + 0.5 * ficr
    print(f"[그룹{g}] NMAE={nmae:.4f}  FICR={ficr:.4f}  총점={score:.4f}")

In [ ]:
# ScoreLossFunction 동작 확인 (그룹별 설비용량으로 초기화)
loss_fns = {
    g: ScoreLossFunction(capacity_kw=RATED_CAPACITY_KW[g])
    for g in GROUPS
}
# ScoreLossFunction: FICR의 계단함수를 시그모이드로 근사해 PyTorch backward() 가능

dummy_pred   = torch.tensor([100.0, 200.0, 300.0])
dummy_actual = torch.tensor([110.0, 190.0, 310.0])
# 소규모 더미 배치로 손실함수 forward 패스 검증

for g, fn in loss_fns.items():
    loss_val = fn(dummy_pred, dummy_actual)
    print(f"[그룹{g}] loss(1-총점 근사)={loss_val.item():.6f}")

---
## Step 6. 그룹별(KPX 1/2/3) 개별 모형 학습

```
ldaps_group_df ─┐
gfs_group_df  ─┤  WindforceDatasetBuilder.build(group)
train_labels  ─┘         ↓
                   피처+타깃 DataFrame
                         ↓
             GroupExperimentRunner.runGroup(group)
                         ↓
   Persistence / SVR / LSTM 세 모델 학습 & 평가
                         ↓
            EvaluationMetrics → nmae / ficr / score
```

In [ ]:
class WindforceDatasetBuilder:
    """LDAPS·GFS 그룹 집계 피처와 train_labels를 결합해
    그룹별 학습용 DataFrame을 만드는 클래스"""

    def __init__(self, ldaps_group_df, gfs_group_df, labels_df):
        self.ldaps_group_df = ldaps_group_df
        self.gfs_group_df   = gfs_group_df
        self.labels_df      = labels_df

    @staticmethod
    def featureCols(df: pd.DataFrame) -> list:
        exclude = {"kst_dtm", "forecast_kst_dtm"} | {f"kpx_group_{i}" for i in range(1, 4)}
        return [c for c in df.columns if c not in exclude]

    def build(self, group: int) -> pd.DataFrame:
        # group 키는 'kpx_group_1' 형식으로 저장됨
        group_key = f"kpx_group_{group}"

        ldaps_g = self.ldaps_group_df[
            self.ldaps_group_df["group"] == group_key
        ].drop(columns="group", errors="ignore")

        gfs_g = self.gfs_group_df[
            self.gfs_group_df["group"] == group_key
        ].drop(columns="group", errors="ignore")

        merged = pd.merge(
            ldaps_g, gfs_g,
            on="forecast_kst_dtm",
            how="inner",
            suffixes=("_ldaps", "_gfs"),
        )

        labels_g = self.labels_df[["kst_dtm", group_key]].copy()
        labels_g["kst_dtm"] = pd.to_datetime(labels_g["kst_dtm"])
        merged["forecast_kst_dtm"] = pd.to_datetime(merged["forecast_kst_dtm"])

        result = pd.merge(
            merged, labels_g,
            left_on="forecast_kst_dtm",
            right_on="kst_dtm",
            how="left",
        ).drop(columns="kst_dtm", errors="ignore")

        result = result.dropna(subset=[group_key]).reset_index(drop=True)

        num_cols = result.select_dtypes(include="number").columns.tolist()
        return result[num_cols + ["forecast_kst_dtm"]].copy()


In [ ]:
# WindforceDatasetBuilder 인스턴스 생성 및 그룹1 데이터 확인
dataset_builder = WindforceDatasetBuilder(
    ldaps_group_df=ldaps_group_df,
    gfs_group_df=gfs_group_df,
    labels_df=loader["train_labels"],
)

sample_df = dataset_builder.build(1)
print(f"KPX 그룹1 학습 데이터: {sample_df.shape}")
print("피처 컬럼 수:", len(WindforceDatasetBuilder.featureCols(sample_df)))
sample_df.head(3)

In [ ]:
# GroupExperimentRunner: Persistence / SVR / LSTM 3모델을 3그룹에 대해 학습·평가
runner = GroupExperimentRunner(
    datasetBuilder=dataset_builder,
    metrics=metrics,
    groups=GROUPS,
    testRatio=0.2,
    seqLen=SEQ_LEN,
)
# testRatio=0.2 : 뒤 20% 기간을 검증용으로 사용 (시간순 분할, 랜덤 셔플 금지)

result_df = runner.runAll()
# 3그룹 × 3모델 = 9개 결과가 result_df에 저장됨

print("\n=== 모델별 그룹별 결과 (pivot) ===")
print(result_df.pivot(index="model_name", columns="group", values=["nmae", "ficr", "score"]))

In [ ]:
# 대회 정식 총점 계산 (3그룹 평균 기준)
final_scores = runner.finalScores()
print("\n=== 최종 총점 (3그룹 평균) ===")
for model_name, score in sorted(final_scores.items(), key=lambda x: x[1], reverse=True):
    print(f"  {model_name:<12}: {score:.4f}")
# 높을수록 좋음. 0.5 초과가 의미 있는 예측, 0.7 이상이 경쟁력 있는 수준

---
## Step 7. 스키마 검증 + 제출 CSV 저장

평가 기간(2025년) LDAPS·GFS 예보로 3개 그룹 발전량을 예측하고  
`sample_submission.csv` 형식에 맞춰 제출 파일을 생성한다.

In [ ]:
# 평가 기간 LDAPS·GFS 전처리
print("평가 기간 LDAPS 격자 전처리 중...")
ldaps_test_grid  = ldaps_fe.transform(loader["ldaps_test"])
ldaps_test_group = ldaps_fe.transformToGroupIDW(ldaps_test_grid, turbine_meta)
# 학습 때와 동일한 ldaps_fe 인스턴스를 재사용 → IDW 가중치가 이미 계산되어 있어 빠름

print("평가 기간 GFS 격자 전처리 중...")
gfs_test_grid  = gfs_fe.transform(loader["gfs_test"])
gfs_test_group = gfs_fe.transformToGroupIDW(gfs_test_grid, turbine_meta)

print(f"test LDAPS shape: {ldaps_test_group.shape}")
print(f"test GFS shape  : {gfs_test_group.shape}")

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# 가장 좋은 모델(LSTM)로 평가 기간 예측 생성
submission = loader["sample_submission"].copy()
submission["forecast_kst_dtm"] = pd.to_datetime(submission["forecast_kst_dtm"])
# 제출 양식의 forecast_kst_dtm 컬럼 datetime 변환

for g in GROUPS:
    target_col = f"kpx_group_{g}"
    group_key  = f"kpx_group_{g}"  # transformToGroupIDW가 저장하는 그룹 키 형식

    # ── 학습 데이터 준비 ──────────────────────────────
    train_df  = dataset_builder.build(g)
    feat_cols = WindforceDatasetBuilder.featureCols(train_df)
    X_train   = train_df[feat_cols].values
    y_train   = train_df[target_col].values

    # ── 평가 데이터 준비 ──────────────────────────────
    ldaps_g = ldaps_test_group[ldaps_test_group["group"] == group_key].drop(columns="group", errors="ignore")
    gfs_g   = gfs_test_group[gfs_test_group["group"] == group_key].drop(columns="group", errors="ignore")
    test_merged = pd.merge(
        ldaps_g, gfs_g,
        on="forecast_kst_dtm", how="inner",
        suffixes=("_ldaps", "_gfs"),
    )
    test_merged["forecast_kst_dtm"] = pd.to_datetime(test_merged["forecast_kst_dtm"])
    test_num_cols = [c for c in test_merged.select_dtypes(include="number").columns if c in feat_cols]

    X_test  = test_merged[test_num_cols].values
    y_dummy = np.zeros(len(X_test))

    # ── 스케일링 (train fit → test transform) ─────────
    scaler     = MinMaxScaler().fit(X_train[:, :len(test_num_cols)])
    X_train_sc = scaler.transform(X_train[:, :len(test_num_cols)])
    X_test_sc  = scaler.transform(X_test)

    # ── LSTM 학습 + 예측 ──────────────────────────────
    pipeline = LSTMPipeline(capacity_kw=RATED_CAPACITY_KW[g], seq_len=SEQ_LEN)
    pipeline.fit(X_train_sc, y_train)
    pred_arr = pipeline.predict(X_test_sc, y_dummy)

    # ── 제출 양식에 예측값 채우기 ─────────────────────
    pred_dtm    = test_merged["forecast_kst_dtm"].iloc[SEQ_LEN:].values
    pred_series = pd.Series(pred_arr, index=pd.to_datetime(pred_dtm))

    submission_idx       = submission.set_index("forecast_kst_dtm").index
    submission[target_col] = pred_series.reindex(submission_idx).values

    print(f"[그룹{g}] 예측값 생성 완료. NaN 수: {submission[target_col].isna().sum()}")

# NaN이 있으면 직전값으로 채움
submission = submission.ffill().fillna(0)
print("\n=== 제출 파일 미리보기 ===")
print(submission.head(5))


In [ ]:
# ── 스키마 검증 ────────────────────────────────────────
sample = loader["sample_submission"]
assert list(submission.columns) == list(sample.columns), "컬럼 순서 불일치!"
assert len(submission) == len(sample), f"행 수 불일치! 기대: {len(sample)}, 실제: {len(submission)}"
assert submission.isna().sum().sum() == 0, "제출 파일에 NaN 존재!"
for g in GROUPS:
    col = f"kpx_group_{g}"
    assert (submission[col] >= 0).all(), f"{col}에 음수 예측값 존재!"
print("스키마 검증 통과 ✅")

# ── 저장 ───────────────────────────────────────────────
out_path = f"{ROOT}/submission_baseline3.csv"
submission.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"제출 파일 저장 완료: {out_path}")


In [ ]:
# print(loader["sample_submission"].columns.tolist())
# print(loader["sample_submission"].head(2))